# Live Demo: Function Calling with the Conversations API

A complete, end-to-end **function-calling loop** built on `client.conversations.create()`. We define a few example tools, let the model decide which to call, execute them locally, feed the results back into the **same conversation**, and get the final answer.

**The round-trip:**
1. Create a durable conversation.
2. Send a user turn + tool definitions.
3. Model emits one or more `function_call` items.
4. We execute each tool locally and append `function_call_output` items.
5. Send those outputs back (same conversation) -> model produces the final answer.

> **Model:** `gpt-5.6-sol` (the bare `gpt-5.6` alias also routes to Sol) — tool orchestration is reasoning-heavy. `reasoning_effort` is pinned to `medium`.

In [1]:
from openai import OpenAI
import json

client = OpenAI()

## 1. Define example tools

In [2]:
# Three example tools the model can call.
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. 'Lisbon'"},
                "unit": {"type": "string", "enum": ["c", "f"], "description": "Temperature unit"},
            },
            "required": ["city"],
        },
    },
    {
        "type": "function",
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string", "description": "ISO code, e.g. 'USD'"},
                "to_currency": {"type": "string", "description": "ISO code, e.g. 'EUR'"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
    {
        "type": "function",
        "name": "get_time",
        "description": "Get the current local time in a city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
]

## 2. Implement the tools (local execution)

In [3]:
# Real implementations would call external APIs; here we mock them deterministically.
def get_weather(city, unit="c"):
    temp_c = {"lisbon": 22, "london": 14, "tokyo": 18}.get(city.lower(), 20)
    temp = temp_c if unit == "c" else round(temp_c * 9 / 5 + 32)
    return {"city": city, "temp": temp, "unit": unit, "conditions": "partly cloudy"}

def convert_currency(amount, from_currency, to_currency):
    rates = {("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09, ("USD", "JPY"): 156.0}
    rate = rates.get((from_currency.upper(), to_currency.upper()), 1.0)
    return {"amount": amount, "from": from_currency, "to": to_currency,
            "rate": rate, "converted": round(amount * rate, 2)}

def get_time(city):
    return {"city": city, "local_time": "14:35", "tz": "local-mock"}

# Dispatch table: tool name -> callable
TOOL_IMPL = {
    "get_weather": get_weather,
    "convert_currency": convert_currency,
    "get_time": get_time,
}

def run_tool(name, arguments_json):
    args = json.loads(arguments_json)
    result = TOOL_IMPL[name](**args)
    return json.dumps(result)

## 3. The function-calling loop (Conversations API)

In [4]:
# Create a durable conversation that carries tool calls + outputs across turns.
conversation = client.conversations.create()

user_message = (
    "What's the weather in Lisbon, and how much is 100 USD in EUR? "
    "Also what's the local time there?"
)

def run_function_calling_turn(user_text, max_rounds=5):
    """Drive the model<->tools loop until the model stops asking for tools."""
    # First model turn with the user's request.
    response = client.responses.create(
        model="gpt-5.6-sol",
        conversation=conversation.id,
        tools=tools,
        input=[{"role": "user", "content": user_text}],
        reasoning={"effort": "medium"},
    )

    for round_i in range(max_rounds):
        # Collect any function calls the model emitted this round.
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response  # model is done -> final answer is ready

        # Execute each requested tool and build the outputs to send back.
        tool_outputs = []
        for call in calls:
            print(f"  -> model called {call.name}({call.arguments})")
            output = run_tool(call.name, call.arguments)
            tool_outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": output,
            })

        # Send tool outputs back on the SAME conversation; model continues reasoning.
        response = client.responses.create(
            model="gpt-5.6-sol",
            conversation=conversation.id,
            tools=tools,
            input=tool_outputs,
            reasoning={"effort": "medium"},
        )

    return response  # safety stop after max_rounds

final = run_function_calling_turn(user_message)
print("\n=== FINAL ANSWER ===")
print(final.output_text)

  -> model called get_weather({"city":"Lisbon","unit":"c"})
  -> model called convert_currency({"amount":100,"from_currency":"USD","to_currency":"EUR"})
  -> model called get_time({"city":"Lisbon"})



=== FINAL ANSWER ===
- **Weather in Lisbon:** 22°C, partly cloudy  
- **100 USD:** approximately **€92.00**  
- **Local time in Lisbon:** **14:35**


## 3.5. OpenAI's Own Answer: Programmatic Tool Calling

The loop above is a **general, vendor-agnostic mechanic** -- you'll rebuild some version of it against any tool-calling LLM API, so it's worth knowing by hand. But the Responses API now ships a server-side alternative to exactly that pattern: **Programmatic Tool Calling**, GA since the GPT-5.6 launch (2026-07-09).

Instead of you shuttling `function_call` -> execute locally -> `function_call_output` back and forth over the wire, the model writes a short JavaScript program that OpenAI runs server-side, in a fresh, isolated V8 sandbox: no Node.js, no package installs, no direct network access, no general-purpose filesystem, no subprocess execution, no state persisted between programs. The program can call your eligible tools directly -- in parallel, with loops and conditionals -- and only hands control back to you once it has something worth surfacing. OpenAI reports named-customer round-trip token reductions of **38-63.5%** versus the manual loop pattern for workloads that fit this shape [source: marktechpost.com GPT-5.6 launch coverage, 2026-07-09] [source: developers.openai.com/api/docs/guides/tools-programmatic-tool-calling].

**Keep the manual loop as your primary pattern.** It's transferable to any provider and gives you full control over every round trip -- that's still the mechanic to teach first. Reach for Programmatic Tool Calling as an *optimization* when a stage has predictable control flow and the code can return a smaller, structured result than a chain of individual tool calls would.

To enable it: mark eligible tools with `"allowed_callers": ["programmatic"]` (typically alongside an `"output_schema"` and `"strict": true`), then add the hosted `{"type": "programmatic_tool_calling"}` tool to your `tools` list. This example reuses `client` and `run_tool()` from above -- it is **verified runnable** against the live API, not pseudocode.

In [5]:
# Programmatic Tool Calling: the model writes JS that calls eligible tools
# server-side, inside an isolated V8 sandbox, instead of round-tripping
# function_call / function_call_output over the wire the way the manual loop does.

ptc_tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. 'Lisbon'"},
                "unit": {"type": "string", "enum": ["c", "f"], "description": "Temperature unit"},
            },
            "required": ["city", "unit"],
            "additionalProperties": False,
        },
        # output_schema lets the sandboxed program reason about the tool's
        # result shape without an extra round trip back to us.
        "output_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string"},
                "temp": {"type": "number"},
                "unit": {"type": "string"},
                "conditions": {"type": "string"},
            },
            "required": ["city", "temp", "unit", "conditions"],
            "additionalProperties": False,
        },
        "allowed_callers": ["programmatic"],  # the sandboxed program may call this directly
        "strict": True,
    },
    {"type": "programmatic_tool_calling"},  # enable the hosted feature
]

ptc_conversation = [
    {"role": "user", "content": "What's the weather in Lisbon and London? Compare them in one sentence."}
]

# Same request/response shape as run_function_calling_turn() above -- any
# function_call items that DO surface (e.g. for tools without
# allowed_callers=["programmatic"]) still go through our local run_tool()
# dispatcher exactly as before.
for _ in range(6):
    response = client.responses.create(
        model="gpt-5.6-sol",
        store=False,
        input=ptc_conversation,
        tools=ptc_tools,
    )
    ptc_conversation.extend(item.model_dump(exclude_none=True) for item in response.output)

    calls = [item for item in response.output if item.type == "function_call"]
    if not calls:
        print(response.output_text)
        break

    for call in calls:
        output = run_tool(call.name, call.arguments)
        ptc_conversation.append({
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": output,
            "caller": call.caller.model_dump() if getattr(call, "caller", None) else None,
        })

Lisbon is partly cloudy at 22°C, while London is also partly cloudy but cooler at 14°C.


## 4. What just happened

- We created one **durable conversation** and never rebuilt history by hand.
- The model emitted `function_call` items; we executed them locally and replied with `function_call_output` items keyed by `call_id`.
- The loop repeats until the model stops requesting tools — handling **parallel** and **multi-round** tool use automatically.
- Because everything is attached to the conversation id, the tool calls and outputs persist and stay in context for any follow-up turn.

Try a follow-up on the same conversation to see the persisted context in action:

In [6]:
# Follow-up turn reuses the SAME conversation -- prior tool results are still in context.
followup = client.responses.create(
    model="gpt-5.6-sol",
    conversation=conversation.id,
    tools=tools,
    input=[{"role": "user", "content": "Convert that same USD amount to JPY instead."}],
    reasoning={"effort": "medium"},
)
# It may call convert_currency again; drive one more loop if needed.
calls = [i for i in followup.output if i.type == "function_call"]
if calls:
    outs = [{"type": "function_call_output", "call_id": c.call_id,
             "output": run_tool(c.name, c.arguments)} for c in calls]
    followup = client.responses.create(
        model="gpt-5.6-sol", conversation=conversation.id, tools=tools,
        input=outs, reasoning={"effort": "medium"},
    )
print(followup.output_text)

**100 USD ≈ ¥15,600 JPY.**
